# 08 — UAE-domain fine-tune (CarDD + VehiDE + UAE pseudo-labels)

**Run on Kaggle with GPU** (Settings → Accelerator → GPU T4 x2 or P100; free 30 hrs/week).

**Why this notebook exists.** The shipped detector (notebook 02, mAP@0.5 = 0.732) was trained on
CarDD+VehiDE — curated, mostly-Western photos. Real UAE listing photos are a different distribution:
harsh overhead sun, dust film, whole-car framing, showroom floors. Its weakest classes are exactly
the ones users photograph most (dent recall 0.53, scratch 0.53, crack 0.37 on the *clean* val set —
certainly worse in the wild). This notebook adapts it with:

1. **UAE pseudo-labeled data** built by `scripts/build_uae_cv_set.py` from real Dubizzle listing
   photos (high-confidence detections become labels; a held-out UAE val split stays untouched).
2. **Heavy sun/quality augmentation** — HSV shifts toward harsh light, mosaic, mixup, copy-paste —
   so the model stops treating UAE lighting as out-of-distribution.
3. **Honest dual eval** — mAP on the original CarDD/VehiDE val AND on the UAE val split, reported
   separately, *before and after*. The UAE delta is the domain-gap number; the CarDD number guards
   against catastrophic forgetting. Both go into `eval/cv_eval_report.json` / the model card.

**Inputs**
- Kaggle dataset with the combined CarDD+VehiDE set from notebook 01 (same as notebook 02 used).
- The `uae_cv_set/` folder produced by `scripts/build_uae_cv_set.py` uploaded as a Kaggle dataset
  (zip `data/raw/uae_cv_set/` and upload; it is gitignored, so it never lives in the repo).
- The current weights `cv-service/model/best.pt` (or start from `yolov8s.pt` if unavailable).

**Acceptance (master plan A1/A2):** UAE-val mAP@0.5 reported honestly; CarDD-val mAP within ~2pts
of 0.732; exported ONNX < 50 MB. If UAE mAP does not improve over the baseline eval, say so in the
model card — a null result on pseudo-labels is a finding, not a failure.

In [ ]:
# Kaggle may assign a Tesla P100 (sm_60), which the preinstalled torch 2.10 no longer
# builds kernels for -> "no kernel image available". Install a torch that still does.
import subprocess, torch
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 7:
    subprocess.run(["pip", "install", "-q", "torch==2.4.1", "torchvision==0.19.1",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### Locate both datasets and build a merged `data.yaml`
CarDD+VehiDE from the notebook-01 dataset; UAE pseudo-labels from your uploaded `uae_cv_set` dataset.
YOLO accepts multiple train dirs in one yaml — the merge happens here, not on disk.

In [ ]:
import glob, json, yaml
from pathlib import Path

UNIFIED = ["dent","scratch","crack","glass_shatter","lamp_broken","tire_flat","punctured","missing_part"]

def find_dir(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"missing {pattern} — attach the dataset to this notebook"
    return Path(hits[0])

cardd_train = find_dir("images/train")          # from the notebook-01 combined dataset
uae_root    = find_dir("uae_cv_set/images/train").parents[1]
print("CarDD+VehiDE:", cardd_train.parents[1], "| UAE:", uae_root)

n_uae_train = len(list((uae_root/"images/train").glob("*.jpg")))
n_uae_val   = len(list((uae_root/"images/val").glob("*.jpg")))
print(f"UAE images: {n_uae_train} train / {n_uae_val} val (pseudo-labeled, listing-level split)")

DATA = {
    "names": dict(enumerate(UNIFIED)),
    "train": [str(cardd_train), str(uae_root/"images/train")],
    "val": str(cardd_train.parent/"val"),          # primary val = original gold labels
}
DATA_YAML = "/kaggle/working/data_uae.yaml"
Path(DATA_YAML).write_text(yaml.safe_dump(DATA))
# separate yaml for the UAE-only eval so the two numbers can never blur together
UAE_YAML = "/kaggle/working/data_uae_val.yaml"
Path(UAE_YAML).write_text(yaml.safe_dump({"names": dict(enumerate(UNIFIED)),
                                          "train": str(uae_root/"images/train"),
                                          "val": str(uae_root/"images/val")}))

### Baseline: current shipped weights on BOTH val sets (before any training)
Without this cell the "improvement" claim is unfalsifiable. Attach the repo's `best.pt` as a
dataset if you have it; otherwise the baseline is the COCO-pretrained `yolov8s.pt` and the CarDD
number from `eval/cv_eval_report.json` (0.732) stands as the shipped baseline.

In [ ]:
from ultralytics import YOLO

prev = sorted(glob.glob("/kaggle/input/**/best.pt", recursive=True))
BASE_W = prev[0] if prev else "yolov8s.pt"
print("baseline weights:", BASE_W)
baseline = {}
if prev:
    m0 = YOLO(BASE_W)
    baseline["cardd_val_mAP50"] = float(m0.val(data=DATA_YAML, plots=False).box.map50)
    baseline["uae_val_mAP50"]   = float(m0.val(data=UAE_YAML,  plots=False).box.map50)
    print(json.dumps(baseline, indent=2))

### Train — UAE-hardened augmentation
The aug block is the domain adaptation: strong HSV-V (harsh sun / deep shadow), HSV-S (dust
desaturation), mosaic + mixup + copy-paste (small-damage recall — the dent/scratch weakness),
and slight perspective (walk-around phone angles). Pseudo-labels are noisy, so `label_smoothing`
takes the edge off confident-wrong boxes.

In [ ]:
model = YOLO(BASE_W)
results = model.train(
    data=DATA_YAML,
    epochs=40, patience=10,
    imgsz=640, batch=-1,
    seed=42, deterministic=True,
    # domain-adaptation augmentation
    hsv_h=0.02, hsv_s=0.55, hsv_v=0.55,
    mosaic=1.0, mixup=0.15, copy_paste=0.20,
    degrees=4.0, perspective=0.0004, fliplr=0.5,
    label_smoothing=0.05,
    project="/kaggle/working/runs", name="uae_finetune",
)
BEST = '/kaggle/working/runs/uae_finetune/weights/best.pt'

### Honest dual eval — the two numbers that matter
- **CarDD/VehiDE val**: must stay within ~2 pts of 0.732 (no catastrophic forgetting).
- **UAE val**: the domain-gap number, vs the baseline cell above. Report per-class too —
  dent/scratch/crack are the classes this whole exercise is for.

In [ ]:
m = YOLO(BEST)
after = {}
r1 = m.val(data=DATA_YAML, plots=False); after["cardd_val_mAP50"] = float(r1.box.map50)
r2 = m.val(data=UAE_YAML,  plots=False); after["uae_val_mAP50"]   = float(r2.box.map50)
per_class = {UNIFIED[i]: round(float(v), 4) for i, v in zip(r1.box.ap_class_index, r1.box.ap50)}
report = {"baseline": baseline, "after": after, "per_class_cardd": per_class,
          "uae_train_images": n_uae_train, "uae_val_images": n_uae_val,
          "note": "UAE val is pseudo-label-derived until the review queue is hand-verified; treat its absolute value with care — the BEFORE/AFTER DELTA on the same split is the meaningful number."}
print(json.dumps(report, indent=2))
Path("/kaggle/working/uae_finetune_report.json").write_text(json.dumps(report, indent=2))

### Export ONNX for the browser + backend
Same export settings as notebook 02 — the app consumes `[1,12,8400]` fp32 at 640².
Download both files and drop them into the repo:
- `best.onnx` → `cv-service/model/best.onnx` AND `frontend/public/models/best.onnx`
- `uae_finetune_report.json` → merge into `eval/cv_eval_report.json` (keep both val numbers)

Then run the full eval suite + Playwright locally before committing — the browser decode
asserts the exact output shape, and a changed class order would silently mislabel damage.

In [ ]:
m.export(format="onnx", imgsz=640, opset=12, simplify=True)
import shutil, os
onnx_path = BEST.replace("best.pt", "best.onnx")
size_mb = os.path.getsize(onnx_path) / 1e6
assert size_mb < 50, f"ONNX is {size_mb:.0f} MB — too heavy for the in-browser path"
shutil.copy(onnx_path, "/kaggle/working/best.onnx")
print(f"exported /kaggle/working/best.onnx ({size_mb:.1f} MB)")